### 12 — Explainable AI Foundations

#### 1. What is Explainable AI?

normal ML workflow:

``` text

Input
  ↓
Model
  ↓
Prediction

```

For CNN project:

``` text

Casting image
     ↓
Fine-Tuned ResNet18
     ↓
def_front / ok_front

```

The model gives us an answer, but that doesn't necessarily tell us why it produced that answer.

Explainable AI adds another question:

``` text

Input
  ↓
Model
  ↓
Prediction
  ↓
WHY did the model make this prediction?
🔍 Explainable AI (XAI)

```

Explainable AI is the collection of methods used to help humans understand the behavior and predictions of AI/ML models.

For example:

``` text

Image
  ↓
ResNet18
  ↓
Prediction: def_front
  ↓
🔍 XAI
  ↓
Which image regions influenced
the defect prediction?

```

That last question is what we'll eventually investigate with Grad-CAM.

#### 2. Why do ML and AI models need explanations?

Consider models you've already worked with.

**Logistic Regression**

Conceptually:

``` text

features
   ↓
feature × coefficient
   ↓
sum + intercept
   ↓
prediction

```

The coefficients are directly inspectable.

Suppose:

- Contract_MonthToMonth    +1.8
- Tenure                   -0.7
- TwoYearContract          -1.4

The exact interpretation depends on preprocessing and the model formulation, but we can at least inspect how the model mathematically uses its features.

**Decision Tree**

A tree may be inspectable as a sequence of decisions:

``` text

Contract = Month-to-month?
          │
        Yes
          ↓
Tenure < 12?
          │
        Yes
          ↓
Predict Churn

```

Again, relatively understandable.

Now compare that with our ResNet18:

``` text

Image
 ↓
Conv
 ↓
Residual blocks
 ↓
thousands of intermediate activations
 ↓
512 learned features
 ↓
Linear(512 → 2)
 ↓
Prediction

```

There isn't a simple human-readable rule such as:

IF scratch > 3 cm
THEN defect

The model learned its own internal representation.

That's why complex models are often described informally as black boxes.

**3. What does "black box" mean?**

A black-box model is not literally unknowable.

We know the architecture:

``` text

input
 ↓
conv1
 ↓
layer1
 ↓
layer2
 ↓
layer3
 ↓
layer4
 ↓
avgpool
 ↓
fc
 ↓
prediction

```

We also have access to its weights.

The problem is that millions of numerical operations do not naturally translate into a simple human explanation.

For example, suppose ResNet produces:

- def_front probability = 0.0154
- ok_front probability  = 0.9846

We understand the calculation mechanically.

But we still don't know:

What visual evidence in this particular casting image caused the model to strongly prefer ok_front?

That's the explanation problem.

**4. Interpretability vs Explainability**

You'll see both terms frequently.

A useful working distinction is: Interpretability

How understandable is the model itself?

For example:

**Small Decision Tree**

``` text

feature A?
   ↓
feature B?
   ↓
prediction

```

A human may be able to inspect the model directly.

**Explainability**

Can we produce useful information explaining the behavior or predictions of the model?

For example:

``` text

ResNet18
   ↓
complex internal computation
   ↓
Prediction: defect
   ↓
Grad-CAM
   ↓
highlight influential image regions

```

So a useful mental model is:

INTERPRETABILITY --> "Can I understand the model itself?"

EXPLAINABILITY --> "Can I understand why the model behaved the way it did?"

The terms aren't used perfectly consistently across every paper or library, so don't treat this as an absolute dictionary boundary. It's a useful practical distinction.

**5. Global vs Local explanations**

This is an important XAI concept.

**Global explanation** : A global explanation tries to help us understand the model's overall behavior.

Question: What does this model generally rely on when classifying products?

Conceptually:

                    MODEL
                      ↓
        ┌─────────────┼─────────────┐
        ↓             ↓             ↓
 surface texture   boundaries    dark regions

We are studying patterns across the model or many observations.


**Local explanation** : A local explanation asks about one specific prediction.

For example, from Notebook 11:

cast_def_0_150.jpeg

- Actual: def_front
- ResNet prediction: ok_front
- Confidence:98.46%

Our local XAI question becomes:

Why did ResNet18 classify this particular image as OK?

That is exactly the kind of question Grad-CAM can help investigate.

``` text

One image
    ↓
One prediction
    ↓
One Grad-CAM heatmap
    ↓
Local explanation

```

So remember:

GLOBAL --> How does the model generally behave?

LOCAL --> Why did the model make this particular prediction?

Notebook 13 will focus heavily on local explanations.

**6. Prediction, confidence, correctness and explanation**

These four concepts must remain separate.

Our real example makes this very clear.

Imagine:

- Actual class: def_front
- Model prediction:ok_front
- Softmax confidence:98.46%

We have three pieces of information already.

- Prediction :  What did the model choose? → ok_front
- Confidence : How strongly did the model prefer its predicted class? → 98.46% softmax probability
- Correctness : Did prediction match ground truth? → No

But we're still missing:

Explanation : What influenced the model to choose ok_front? → not answered yet

Therefore:

- Prediction ≠ Confidence
- Confidence ≠ Correctness
- Correctness ≠ Explanation

And especially:

High confidence is not an explanation.

A model can be:

- high confidence + correct
- high confidence + wrong
- low confidence + correct
- low confidence + wrong

Our 98.46%-confidence false negative is an excellent example of the second case.

**7. What is feature importance?**


Suppose a churn model uses:

- tenure
- contract
- monthly charges
- internet service
- payment method
- ...

We may ask:

Which features had the greatest influence on the model?

That's a feature-importance-style question.

But images are different.

Our input isn't:

- tenure = 12
- contract = monthly
- charges = 80

It's approximately: [3, 224, 224]

containing thousands of pixel values arranged spatially.

And deeper CNN layers transform those pixels into learned feature maps.

Therefore, for computer vision we often want an explanation that preserves spatial meaning: Where in the image was influential?

That's why heatmaps are intuitive for image XAI.

**8. From feature importance to image regions**

Suppose this is our casting:

``` text

+--------------------------+
|                          |
|        casting           |
|                          |
|             defect       |
|               ●          |
|                          |
+--------------------------+

Imagine the model predicts: def_front

We'd like an XAI technique to indicate something conceptually like:

+--------------------------+
|                          |
|        casting           |
|                          |
|            █████         |
|            █ ● █         |
|            █████         |
|                          |
+--------------------------+

```

The highlighted region indicates: This area was influential for the model's prediction.

That's the basic intuition behind the visualization we're heading toward.

**9. What is Grad-CAM?**

Grad-CAM stands for: Gradient-weighted Class Activation Mapping

Don't worry about the mathematics yet.

Conceptually, Grad-CAM asks: Which spatial regions represented in a CNN's feature maps were influential for a particular class prediction?

Very simplified:

``` text

Image
  ↓
CNN
  ↓
feature maps
  ↓
class prediction
  ↓
gradients for that class
  ↓
determine influential feature maps
  ↓
combine them
  ↓
heatmap
  ↓
overlay on original image

```

Result:

``` text

Original image
      +
Grad-CAM heatmap
      ↓
Highlighted influential regions

```

We'll unpack each part in Notebook 13.

**10. Why use a later convolutional layer?**

This connects directly to what you learned about CNNs and ResNet.

Earlier layers tend to represent more basic visual information:

``` text

early layers
   ↓
edges
lines
simple patterns

Deeper convolutional layers tend to contain more task-relevant, higher-level representations:

later layers
   ↓
textures
shapes
complex patterns

At the same time, Grad-CAM needs spatial feature maps.

Remember ResNet structure:

conv1
 ↓
layer1
 ↓
layer2
 ↓
layer3
 ↓
layer4       ← spatial feature maps still exist
 ↓
avgpool
 ↓
flatten
 ↓
fc

``` 
Once we reach: avgpool → flatten → fc

much of the explicit spatial arrangement has been collapsed.

That's why Grad-CAM commonly targets a later convolutional block such as: model.layer4

And remember : layer4 is the final major feature-extraction block, not the final layer of ResNet18.

The final classifier is still: model.fc

**11. A Grad-CAM heatmap does NOT show the ground-truth defect**

This is extremely important.

Suppose Grad-CAM highlights: top-right portion of casting

We can say: That region was influential in the model's prediction.

We should not automatically say: Grad-CAM found the defect.

Why?

Because our dataset contains only image-level labels:

image → def_front
image → ok_front

It does not contain: defect bounding box or pixel-level defect mask

Therefore:

``` text

Grad-CAM
   ↓
model-influential region

NOT

ground-truth defect location

```

That's one of the most important interpretation rules for Notebook 13.

**12. Explanations do not prove that the model is correct**

Suppose Grad-CAM highlights what looks like a scratch.

That doesn't automatically prove: model reasoning is correct

Likewise, suppose the heatmap focuses on:

- image border
- background
- lighting artifact
- camera mark

That may reveal a potential shortcut or spurious cue worth investigating.

For example:

``` text

Training data

Defective products
often photographed
with dark border
       ↓
model learns
dark border → defect
       ↓
high accuracy
       ↓
production camera changes
       ↓
model fails

``` 

XAI can help us discover hypotheses like this.

But XAI itself doesn't automatically prove causality or guarantee reliability.

**13. 🔍 XAI and ⚖️ RAI are related, but different**

This distinction is now especially useful.

**🔍 XAI asks**

Why did the model produce this prediction?

Example:

- Why did ResNet classify this defect as OK?
- Which image regions influenced it?

**⚖️ RAI asks**

Broader questions such as:

- Is this model reliable enough for this use?
- What happens when it misses a defect?
- Is the evaluation trustworthy?
- Are there problematic data-quality issues?
- When should a human inspect the product?

Our project already gives examples of both:

``` text

64 exact development/test duplicates
             ↓
⚖️ RAI
evaluation integrity / data quality


98.46%-confidence wrong prediction
             ↓
⚖️ RAI
reliability / confidence concern


Which regions caused that
98.46%-confidence prediction?
             ↓
🔍 XAI
Grad-CAM investigation

And sometimes they interact:

🔍 XAI
finds model focusing on irrelevant background
                ↓
⚖️ RAI
raises reliability concern
                ↓
investigate data/model
                ↓
possible human or engineering action

```

So XAI can provide evidence that contributes to a broader Responsible AI assessment.


#### Key Learnings

- Explainable AI (XAI) helps humans investigate the behavior and
  predictions of machine-learning and AI models.

- XAI is not limited to neural networks or computer vision.
  Explainability techniques can also be applied to traditional ML,
  NLP, transformers, and other AI systems.

- Simple models may be relatively interpretable directly, while complex
  models such as CNNs and transformers often require additional
  explanation techniques.

- Interpretability focuses on how understandable the model itself is,
  while explainability focuses on understanding model behavior or
  particular predictions.

- Global explanations investigate overall model behavior.

- Local explanations investigate individual predictions.

- Prediction, confidence, correctness, and explanation are different
  concepts.

- High softmax confidence does not guarantee that a prediction is
  correct and does not explain why the prediction was made.

- For image classification, spatial explanation methods can help
  identify image regions that influenced a model prediction.

- Grad-CAM uses class-specific gradient information together with CNN
  feature maps to produce a spatial heatmap.

- Later convolutional feature maps are useful for Grad-CAM because they
  retain spatial information while representing higher-level visual
  features.

- A Grad-CAM heatmap shows regions that influenced the model. It does
  not provide ground-truth defect localization.

- An explanation does not prove that a model is correct, causal,
  reliable, or safe.

- 🔍 XAI investigates why a model behaved in a particular way.

- ⚖️ Responsible AI (RAI) uses broader evidence, including model
  performance, data quality, failure cases, explanations, and operational
  consequences, to assess whether a model can be used appropriately.


#### Conclusion

This notebook introduced the foundations of Explainable AI and connected
them to the Product Defect Computer Vision project.

The fine-tuned ResNet18 achieved strong classification performance, but
performance metrics and softmax probabilities alone cannot explain why
individual predictions were made.

The three non-duplicate defect images misclassified by both the custom
CNN and ResNet18 provide useful failure cases for explanation analysis.
One of these images was classified as OK by ResNet18 with approximately
98.46% softmax confidence.

The next notebook will implement Grad-CAM to investigate which spatial
regions of casting images influenced ResNet18 predictions, while
carefully distinguishing model explanations from ground-truth defect
locations.